In [ ]:
# Databricks notebook source
# MAGIC %md
# MAGIC # Trade Area Feature Aggregation
# MAGIC
# MAGIC Aggregates H3 features (from CARTO marketplace) to trade area level:
# MAGIC 1. H3 index trade areas → get h3_cell_ids
# MAGIC 2. Join to h3_features_carto (CARTO data) on h3_cell_id
# MAGIC 3. Aggregate by store_number

In [ ]:
from pyspark.sql import functions as F
import yaml

dbutils.widgets.text("catalog", "jdub_demo_aws")
dbutils.widgets.text("silver_schema", "geo_silver")
dbutils.widgets.text("gold_schema", "geo_gold")
dbutils.widgets.text("config_path", "/Workspace/resources/configs/h3_features_config.yml")
dbutils.widgets.text("trade_area_table", "", "Trade Area Table (optional)")
dbutils.widgets.text("output_table_override", "", "Output Table (optional)")

catalog = dbutils.widgets.get("catalog")
silver_schema = dbutils.widgets.get("silver_schema")
gold_schema = dbutils.widgets.get("gold_schema")
config_path = dbutils.widgets.get("config_path")
trade_area_table_override = dbutils.widgets.get("trade_area_table")
output_table_override = dbutils.widgets.get("output_table_override")

with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

H3_RESOLUTION = config['h3_grid']['resolution']

if trade_area_table_override and trade_area_table_override.strip():
    trade_area_table = trade_area_table_override.strip()
    # Extract table name from full path if provided
    table_name = trade_area_table.split('.')[-1]
    default_output = f"{table_name}_enriched"
else:
    trade_area_table = f"{catalog}.{silver_schema}.lce_isochrones_5min"
    default_output = "lce_trade_area_features"

if output_table_override and output_table_override.strip():
    output_table_name = output_table_override.strip()
else:
    output_table_name = default_output

print(f"Input: {trade_area_table}")
print(f"Output: {catalog}.{gold_schema}.{output_table_name}")

%md
## H3 Index Trade Areas

In [ ]:
# H3 Index Trade Areas - Optimized Hierarchical Approach
from pyspark.sql.functions import col, expr, explode, monotonically_increasing_id, coalesce, lit

# Load trade areas from silver table
trade_areas = spark.table(trade_area_table)

# Flexible column mapping to handle missing columns gracefully
columns = trade_areas.columns
id_col = next((c for c in columns if c in ['store_number', 'point_id', 'id', 'location_id']), None)
lat_col = next((c for c in columns if c in ['latitude', 'lat', 'y']), None)
lon_col = next((c for c in columns if c in ['longitude', 'lon', 'lng', 'x']), None)
type_col = next((c for c in columns if c in ['store_type', 'type', 'category']), None)
city_col = next((c for c in columns if c in ['city', 'municipality']), None)
state_col = next((c for c in columns if c in ['state', 'region']), None)
dt_col = next((c for c in columns if c in ['drive_time_minutes', 'drive_time']), None)
area_col = next((c for c in columns if c in ['area_sqkm', 'area']), None)

# Base selection with aliasing
base_ta = trade_areas.select(
    (col(id_col) if id_col else monotonically_increasing_id().cast("string")).alias("store_number"),
    (col(lat_col) if lat_col else lit(None)).alias("latitude"),
    (col(lon_col) if lon_col else lit(None)).alias("longitude"),
    (col(type_col) if type_col else lit(None)).alias("store_type"),
    (col(city_col) if city_col else lit(None)).alias("city"),
    (col(state_col) if state_col else lit(None)).alias("state"),
    (col(dt_col) if dt_col else lit(None)).alias("drive_time_minutes"),
    (col(area_col) if area_col else lit(None)).alias("area_sqkm"),
    col("geometry")
)

# Optimized Hierarchical H3 Generation (Avoids OOM on large isochrones)
# 1. Coarse cover (Res 5)
# 2. Explode to target resolution
# 3. Filter using Point-in-Polygon
ta_exploded = base_ta.alias("ta").withColumn(
    "coarse_h3", 
    explode(expr("h3_coverash3string(ST_AsBinary(ta.geometry), 5)"))
).select(
    "ta.*",
    explode(expr(f"h3_tochildren(coarse_h3, {H3_RESOLUTION})")).alias("h3_cell_id")
)

ta_h3 = ta_exploded.alias("exploded").join(
    base_ta.select("store_number", "geometry").alias("orig"),
    expr("orig.store_number = exploded.store_number AND ST_Contains(orig.geometry, ST_GeomFromWKT(h3_centeraswkt(exploded.h3_cell_id), 4326))"),
    "inner"
).select(
    col("exploded.store_number"),
    col("exploded.latitude"),
    col("exploded.longitude"),
    col("exploded.store_type"),
    col("exploded.city"),
    col("exploded.state"),
    col("exploded.drive_time_minutes"),
    col("exploded.area_sqkm"),
    col("exploded.geometry"),
    col("exploded.h3_cell_id")
)

print(f"Trade areas indexed with H3")
display(ta_h3.limit(5))

In [ ]:
display(trade_areas)

%md
## Join to H3 Features

In [ ]:
h3_features = spark.table(f"{catalog}.{gold_schema}.h3_features_carto").drop("h3_geometry", "h3_resolution", "processing_timestamp")

ta_with_features = ta_h3.join(h3_features, "h3_cell_id", "inner")

print(f"Joined trade areas with CARTO H3 features")
display(ta_with_features.limit(5))

%md
## Aggregate by Store

In [ ]:
# Get demographic variables from config (CARTO columns)
demo_vars = config.get('carto_demographic_variables', config.get('demographic_variables', {}))
count_vars = (
    demo_vars.get('population', []) + 
    demo_vars.get('income', []) + 
    demo_vars.get('households', []) + 
    demo_vars.get('education', []) + 
    demo_vars.get('employment', []) + 
    demo_vars.get('housing', []) + 
    demo_vars.get('commute', [])
)
# Handle median vars if present (CARTO may have these)
median_vars = demo_vars.get('median', [])

existing_count_vars = [v for v in count_vars if v in ta_with_features.columns]
existing_median_vars = [v for v in median_vars if v in ta_with_features.columns]
# CARTO POI columns (not custom poi_count_ columns)
carto_poi_cols = ['retail', 'education', 'financial', 'food_drink', 'healthcare', 'leisure', 'tourism', 'transportation']
existing_poi_cols = [c for c in carto_poi_cols if c in ta_with_features.columns]
distance_cols = [c for c in ta_with_features.columns if c.startswith('distance_to_')]

In [ ]:
agg_exprs = []

# Count variables: sum (convert negatives to positive for demo purposes)
for var in existing_count_vars:
    agg_exprs.append(F.abs(F.sum(var)).cast("long").alias(var))

# CARTO POI counts: sum (convert negatives to positive)
for col in existing_poi_cols:
    agg_exprs.append(F.abs(F.sum(col)).cast("long").alias(f"total_{col}_pois"))
if 'total_poi_count' in ta_with_features.columns:
    agg_exprs.append(F.abs(F.sum("total_poi_count")).cast("long").alias("total_poi_count"))


# Median/rate variables: avg (convert negatives to positive)
for var in existing_median_vars:
    agg_exprs.append(F.abs(F.avg(var)).alias(var))
if 'per_capita_income' in ta_with_features.columns:
    agg_exprs.append(F.abs(F.avg("per_capita_income")).alias("per_capita_income"))

# Distance features: min (keep as-is, distances should be positive)
for col in distance_cols:
    agg_exprs.append(F.min(col).alias(col))

# Population density and urbanicity: avg (convert negatives to positive)
if 'urbanicity_score' in ta_with_features.columns:
    agg_exprs.append(F.abs(F.avg("urbanicity_score")).alias("urbanicity_score"))

agg_exprs.extend([
    F.count("h3_cell_id").alias("h3_cell_count"),
    F.first("geometry").alias("geometry")  # Keep the isochrone geometry
])

# Aggregate by store, handling potential nulls from flexible mapping
ta_features_agg = ta_with_features.groupBy(
    "store_number",
    "latitude",
    "longitude",
    "store_type",
    "city",
    "state",
    # "urbanicity_category",
    "drive_time_minutes",
    "area_sqkm"
).agg(*agg_exprs)

display(ta_features_agg.limit(5))

In [ ]:
display(ta_features_agg)

%md
## Write to Gold

In [ ]:
ta_features_final = ta_features_agg.withColumn("processing_timestamp", F.current_timestamp())

numeric_cols = [
    field.name for field in ta_features_final.schema.fields 
    if field.dataType.typeName() in ['long', 'double', 'integer', 'float']
    and field.name not in ['latitude', 'longitude', 'drive_time_minutes', 'area_sqkm']
]
ta_features_final = ta_features_final.fillna(0, subset=numeric_cols)

# Ensure one record per store (deduplication)
# Keep first occurrence if there are any duplicates
from pyspark.sql.window import Window
window_spec = Window.partitionBy("store_number").orderBy(F.desc("processing_timestamp"))
ta_features_final = ta_features_final.withColumn("row_num", F.row_number().over(window_spec)).filter(F.col("row_num") == 1).drop("row_num")

output_table = f"{catalog}.{gold_schema}.{output_table_name}"

print(f"Records to write: {ta_features_final.count()}")

(
    ta_features_final
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(output_table)
)

print(f"Written to {output_table}")

In [ ]:
display(spark.sql(f"""
  SELECT
    COUNT(*) as total_trade_areas,
    ROUND(AVG(area_sqkm), 2) as avg_area_sqkm,
    ROUND(AVG(population), 0) as avg_population,
    ROUND(AVG(total_poi_count), 0) as avg_poi_count,
    ROUND(AVG(h3_cell_count), 0) as avg_h3_cells
  FROM {output_table}
"""))